# 🤖 Clarification Agent Test

**Goal:** Test LLM-powered clarification questions before search.

**Flow:**
```
User: "laptop"
    ↓
ClarificationAgent.generate_questions()
    ↓
Bot: 3 questions with options
    ↓
User answers
    ↓
ClarificationAgent.build_refined_query()
    ↓
Refined Query: "Acer laptop for students under $800"
    ↓
Full Pipeline → Results
```

In [1]:
# Cell 1: Setup & Imports
import os
import sys
import logging

# Change working directory to segment4
os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

print(f"Working directory: {os.getcwd()}")

# Setup logging
logging.basicConfig(level=logging.INFO)
root = logging.getLogger()
root.setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

# Core imports
from typing import List, Optional, Tuple
from pydantic import BaseModel, Field
from openai import OpenAI

print("✅ Setup complete!")

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
✅ Setup complete!


In [2]:
# Cell 2: Define Pydantic Schemas for Structured Output

class ClarificationQuestion(BaseModel):
    """A single clarification question with options."""
    question: str = Field(description="The clarification question in English")
    options: List[str] = Field(
        description="4-6 options for the user to choose from. Last option should always be 'Any' or 'No preference'",
        min_length=3,
        max_length=6
    )


class ClarificationResponse(BaseModel):
    """Response containing clarification questions generated by LLM."""
    product_category: str = Field(description="Detected product category (e.g., 'laptop', 'TV', 'headphones')")
    questions: List[ClarificationQuestion] = Field(
        description="Exactly 3 clarification questions to help refine the search",
        min_length=3,
        max_length=3
    )


class RefinedQuery(BaseModel):
    """Refined search query built from user answers."""
    query: str = Field(description="Optimized search query in English for BestBuy search")
    summary: str = Field(description="Brief summary of what the user is looking for")


print("✅ Pydantic schemas defined!")
print("   - ClarificationQuestion")
print("   - ClarificationResponse")
print("   - RefinedQuery")

✅ Pydantic schemas defined!
   - ClarificationQuestion
   - ClarificationResponse
   - RefinedQuery


In [3]:
# Cell 3: Define ClarificationAgent Class

class ClarificationAgent:
    """
    Agent that generates clarification questions before search.
    
    Uses GPT-5-mini to:
    1. Generate 3 relevant questions based on user's keyword
    2. Build a refined search query from user's answers
    """
    
    MODEL = "gpt-5-mini"
    
    GENERATE_QUESTIONS_PROMPT = """You are a smart shopping assistant. When a user wants to search for a product, 
you need to ask 3 clarification questions to better understand their needs.

RULES:
1. Generate exactly 3 questions relevant to the product type
2. Each question should have 4-6 options
3. The last option should always be "Any" or "No preference"
4. Questions should cover different aspects like: brand, use case, budget, features, size, etc.
5. Adapt questions to the product category (laptop questions differ from skincare questions)
6. Keep questions concise and clear

EXAMPLES:
- For "laptop": Ask about brand preference, primary use (gaming/work/study), budget range
- For "TV": Ask about screen size, smart features, brand preference
- For "headphones": Ask about type (over-ear/in-ear), use case (music/gaming/calls), wireless preference
"""
    
    BUILD_QUERY_PROMPT = """Based on the user's original keyword and their answers to clarification questions,
build an optimized search query for BestBuy.

RULES:
1. Query should be in English
2. Keep it concise - only include relevant keywords
3. If user answered "Any" or "No preference", skip that criterion
4. Include price constraint if user specified a budget (e.g., "under $800")
5. Focus on searchable terms that BestBuy would understand
"""
    
    def __init__(self):
        """Initialize with OpenAI client."""
        self.openai = OpenAI()
        print("ClarificationAgent initialized with GPT-5-mini")
    
    def generate_questions(self, keyword: str) -> ClarificationResponse:
        """
        Generate 3 clarification questions based on user's keyword.
        
        Args:
            keyword: User's initial search keyword (e.g., "laptop")
            
        Returns:
            ClarificationResponse with 3 questions and options
        """
        print(f"🤖 Generating questions for: '{keyword}'")
        
        result = self.openai.chat.completions.parse(
            model=self.MODEL,
            messages=[
                {"role": "system", "content": self.GENERATE_QUESTIONS_PROMPT},
                {"role": "user", "content": f"Product keyword: {keyword}"}
            ],
            response_format=ClarificationResponse
        )
        
        response = result.choices[0].message.parsed
        print(f"✅ Generated {len(response.questions)} questions")
        return response
    
    def build_refined_query(
        self, 
        keyword: str, 
        questions: List[ClarificationQuestion],
        answers: List[str]
    ) -> RefinedQuery:
        """
        Build refined search query from user's answers.
        
        Args:
            keyword: Original keyword
            questions: List of questions that were asked
            answers: User's answers to each question
            
        Returns:
            RefinedQuery with optimized search string
        """
        print(f"🔧 Building refined query...")
        
        # Format Q&A for the prompt
        qa_text = ""
        for i, (q, a) in enumerate(zip(questions, answers), 1):
            qa_text += f"Q{i}: {q.question}\nA{i}: {a}\n\n"
        
        user_message = f"""Original keyword: {keyword}

User's answers:
{qa_text}

Build an optimized search query based on these answers."""
        
        result = self.openai.chat.completions.parse(
            model=self.MODEL,
            messages=[
                {"role": "system", "content": self.BUILD_QUERY_PROMPT},
                {"role": "user", "content": user_message}
            ],
            response_format=RefinedQuery
        )
        
        refined = result.choices[0].message.parsed
        print(f"✅ Refined query: '{refined.query}'")
        return refined


print("✅ ClarificationAgent class defined!")

✅ ClarificationAgent class defined!


In [4]:
# Cell 4: Test Generate Questions
# 📝 Change the keyword to test different product types

# Initialize agent
clarification_agent = ClarificationAgent()

# Test keyword - CHANGE THIS TO TEST DIFFERENT PRODUCTS
TEST_KEYWORD = "laptop"

print(f"\n{'='*60}")
print(f"Testing with keyword: '{TEST_KEYWORD}'")
print(f"{'='*60}\n")

# Generate questions
clarification_response = clarification_agent.generate_questions(TEST_KEYWORD)

print(f"\n📦 Product Category: {clarification_response.product_category}")
print(f"\n{'='*60}")
print("🤖 CLARIFICATION QUESTIONS:")
print(f"{'='*60}")

for i, q in enumerate(clarification_response.questions, 1):
    print(f"\n📌 Question {i}: {q.question}")
    print(f"   Options:")
    for j, opt in enumerate(q.options, 1):
        print(f"      {j}. {opt}")

ClarificationAgent initialized with GPT-5-mini

Testing with keyword: 'laptop'

🤖 Generating questions for: 'laptop'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Generated 3 questions

📦 Product Category: laptop

🤖 CLARIFICATION QUESTIONS:

📌 Question 1: Preferred brand or ecosystem?
   Options:
      1. Apple (MacBook)
      2. Dell/HP/Lenovo (business/mainstream)
      3. Asus/MSI/Razer (gaming/creative)
      4. Acer/Other budget brands
      5. No preference

📌 Question 2: Primary use for the laptop?
   Options:
      1. Gaming (high-performance/graphics)
      2. Work/business (office apps, video calls)
      3. Creative work (photo/video editing, design)
      4. Everyday use & study (web browsing, streaming)
      5. Any

📌 Question 3: What's your budget range?
   Options:
      1. Under $500
      2. $500–$1000
      3. $1000–$1500
      4. $1500+
      5. Any


In [5]:
# Cell 5: User Input Answers (Simulate)
# 📝 Type your answers here - can be any text or choose from options above

print("="*60)
print("📝 ENTER YOUR ANSWERS")
print("="*60)
print("(You can type freely or use one of the options shown above)")
print("(Type 'skip' or 'any' to skip a question)\n")

# Display questions again for reference
for i, q in enumerate(clarification_response.questions, 1):
    print(f"Q{i}: {q.question}")
    print(f"    Options: {', '.join(q.options)}")
    print()

# ============================================
# 👇 ENTER YOUR ANSWERS HERE 👇
# ============================================

USER_ANSWERS = [
    "Acer",           # Answer to Question 1
    "for students",   # Answer to Question 2  
    "under $800",     # Answer to Question 3
]

# ============================================

print("\n" + "="*60)
print("✅ YOUR ANSWERS:")
print("="*60)
for i, (q, a) in enumerate(zip(clarification_response.questions, USER_ANSWERS), 1):
    print(f"Q{i}: {q.question}")
    print(f"A{i}: {a}\n")

📝 ENTER YOUR ANSWERS
(You can type freely or use one of the options shown above)
(Type 'skip' or 'any' to skip a question)

Q1: Preferred brand or ecosystem?
    Options: Apple (MacBook), Dell/HP/Lenovo (business/mainstream), Asus/MSI/Razer (gaming/creative), Acer/Other budget brands, No preference

Q2: Primary use for the laptop?
    Options: Gaming (high-performance/graphics), Work/business (office apps, video calls), Creative work (photo/video editing, design), Everyday use & study (web browsing, streaming), Any

Q3: What's your budget range?
    Options: Under $500, $500–$1000, $1000–$1500, $1500+, Any


✅ YOUR ANSWERS:
Q1: Preferred brand or ecosystem?
A1: Acer

Q2: Primary use for the laptop?
A2: for students

Q3: What's your budget range?
A3: under $800



In [6]:
# Cell 6: Build Refined Query

print("="*60)
print("🔧 BUILDING REFINED QUERY")
print("="*60)

refined_query = clarification_agent.build_refined_query(
    keyword=TEST_KEYWORD,
    questions=clarification_response.questions,
    answers=USER_ANSWERS
)

print(f"\n📊 RESULT:")
print(f"   Original keyword: '{TEST_KEYWORD}'")
print(f"   Refined query:    '{refined_query.query}'")
print(f"   Summary:          {refined_query.summary}")

# Store for next cell
SEARCH_QUERY = refined_query.query
print(f"\n✅ Ready to search with: '{SEARCH_QUERY}'")

🔧 BUILDING REFINED QUERY
🔧 Building refined query...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Refined query: 'Acer laptops for students under $800'

📊 RESULT:
   Original keyword: 'laptop'
   Refined query:    'Acer laptops for students under $800'
   Summary:          Acer laptop suitable for student use (note-taking, web browsing, study) with a budget under $800.

✅ Ready to search with: 'Acer laptops for students under $800'


In [7]:
# Cell 7: Import Pipeline Components
# Now we import the existing BestBuy pipeline components

print("Importing pipeline components...")

from price_agents.bestbuy_deals import (
    ScrapedBestBuyDeal,
    filter_sale_urls,
    scrape_bestbuy_products
)
from price_agents.bestbuy_scanner_agent import (
    BestBuySearchAgent,
    BestBuyScannerAgent
)
from price_agents.deals import Deal, DealSelection, Opportunity

print("✅ Pipeline components imported!")
print("   - BestBuySearchAgent (Brave MCP)")
print("   - filter_sale_urls")
print("   - scrape_bestbuy_products (Playwright)")
print("   - BestBuyScannerAgent (GPT-5-mini)")

Importing pipeline components...
✅ Pipeline components imported!
   - BestBuySearchAgent (Brave MCP)
   - filter_sale_urls
   - scrape_bestbuy_products (Playwright)
   - BestBuyScannerAgent (GPT-5-mini)


In [ ]:
# Cell 8: Step 1 - Search URLs with Refined Query

print("="*60)
print(f"🔍 STEP 1: Searching BestBuy for '{SEARCH_QUERY}'")
print("="*60)

# Initialize search agent
search_agent = BestBuySearchAgent()

# Search with refined query
MAX_URLS = 15
urls = search_agent.search(SEARCH_QUERY, max_urls=MAX_URLS)

print(f"\n✅ Found {len(urls)} product URLs:")
for i, url in enumerate(urls[:5], 1):  # Show first 5
    print(f"   {i}. {url[:80]}...")
if len(urls) > 5:
    print(f"   ... and {len(urls) - 5} more")

INFO:root:[BestBuy Search Agent] BestBuy Search Agent is initializing
INFO:root:[BestBuy Search Agent] BestBuy Search Agent is ready
INFO:root:[BestBuy Search Agent] Searching for: Acer laptops for students under $800


🔍 STEP 1: Searching BestBuy for 'Acer laptops for students under $800'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:[BestBuy Search Agent] Found 15 product URLs



✅ Found 15 product URLs:
   1. https://www.bestbuy.com/product/acer-aspire-3-thin-light-laptop-15-6-full-hd-ips...
   2. https://www.bestbuy.com/product/acer-aspire-lite-15-laptop-15-6-fhd-ips-intel-co...
   3. https://www.bestbuy.com/product/acer-nitro-5-17-3-full-hd-ips-144hz-gaming-lapto...
   4. https://www.bestbuy.com/product/acer-aspire-3-a315-24P-r7vh-slim-laptop--15-6-fu...
   5. https://www.bestbuy.com/product/acer-aspire-3-a315-58-7138-laptop-15-6fhd-laptop...
   ... and 10 more


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [9]:
# Cell 9: Step 2 - Filter Sale URLs

print("="*60)
print(f"🏷️ STEP 2: Filtering {len(urls)} URLs for sale items")
print("="*60)

sale_urls = filter_sale_urls(urls)

print(f"\n✅ Found {len(sale_urls)} products on SALE:")
for i, url in enumerate(sale_urls, 1):
    print(f"   {i}. {url[:80]}...")

if not sale_urls:
    print("\n⚠️ No sale items found. Try a different query or skip filtering.")

🏷️ STEP 2: Filtering 15 URLs for sale items


INFO:price_agents.bestbuy_deals:[1/15] Skip
INFO:price_agents.bestbuy_deals:[2/15] SALE
INFO:price_agents.bestbuy_deals:[3/15] SALE
INFO:price_agents.bestbuy_deals:[4/15] Skip
INFO:price_agents.bestbuy_deals:[5/15] SALE
INFO:price_agents.bestbuy_deals:[6/15] Skip
INFO:price_agents.bestbuy_deals:[7/15] Skip
INFO:price_agents.bestbuy_deals:[8/15] Skip
INFO:price_agents.bestbuy_deals:[9/15] Skip
INFO:price_agents.bestbuy_deals:[10/15] Skip
INFO:price_agents.bestbuy_deals:[11/15] Skip
INFO:price_agents.bestbuy_deals:[12/15] SALE
INFO:price_agents.bestbuy_deals:[13/15] Skip
INFO:price_agents.bestbuy_deals:[14/15] Skip
INFO:price_agents.bestbuy_deals:[15/15] Skip
INFO:price_agents.bestbuy_deals:Filtered 15 URLs → 4 sale URLs



✅ Found 4 products on SALE:
   1. https://www.bestbuy.com/product/acer-aspire-lite-15-laptop-15-6-fhd-ips-intel-co...
   2. https://www.bestbuy.com/product/acer-nitro-5-17-3-full-hd-ips-144hz-gaming-lapto...
   3. https://www.bestbuy.com/product/acer-aspire-3-a315-58-7138-laptop-15-6fhd-laptop...
   4. https://www.bestbuy.com/product/acer-nitro-15-6-fhd-ips-144hz-gaming-laptop-13th...


In [10]:
# Cell 10: Step 3 - Scrape Products with Playwright
# ⚠️ This will open a browser window!

print("="*60)
print(f"📦 STEP 3: Scraping {len(sale_urls)} products with Playwright")
print("="*60)
print("⚠️ Browser window will open...\n")

# Scrape products
scraped_deals = await scrape_bestbuy_products(sale_urls, headless=False)

print(f"\n✅ Scraped {len(scraped_deals)} products:")
for i, deal in enumerate(scraped_deals, 1):
    print(f"\n   [{i}] {deal.title[:60]}...")
    print(f"       💰 ${deal.price}")

📦 STEP 3: Scraping 4 products with Playwright
⚠️ Browser window will open...



INFO:price_agents.bestbuy_deals:[1/4] Scraping: https://www.bestbuy.com/product/acer-aspire-lite-15-laptop-1...
INFO:price_agents.bestbuy_deals:  ✓ <Acer - Aspire Lite 15 Laptop – 15.6" FHD IPS – Int... | $329.99>
INFO:price_agents.bestbuy_deals:[2/4] Scraping: https://www.bestbuy.com/product/acer-nitro-5-17-3-full-hd-ip...
INFO:price_agents.bestbuy_deals:  ✓ <Acer - Nitro 5 17.3" Full HD IPS 144Hz Gaming Lapt... | $279.99>
INFO:price_agents.bestbuy_deals:[3/4] Scraping: https://www.bestbuy.com/product/acer-aspire-3-a315-58-7138-l...
INFO:price_agents.bestbuy_deals:  ✓ <Acer - Aspire 3 A315-58-7138 Laptop 15.6"FHD Lapto... | $638.97>
INFO:price_agents.bestbuy_deals:[4/4] Scraping: https://www.bestbuy.com/product/acer-nitro-15-6-fhd-ips-144h...
INFO:price_agents.bestbuy_deals:  ✓ <Acer - Nitro 15.6\" FHD IPS 144Hz Gaming Laptop -1... | $879.99>
INFO:price_agents.bestbuy_deals:Successfully scraped 4/4 products



✅ Scraped 4 products:

   [1] Acer - Aspire Lite 15 Laptop – 15.6" FHD IPS – Intel Core 3 ...
       💰 $329.99

   [2] Acer - Nitro 5 17.3" Full HD IPS 144Hz Gaming Laptop- Intel ...
       💰 $279.99

   [3] Acer - Aspire 3 A315-58-7138 Laptop 15.6"FHD Laptop,lntel Co...
       💰 $638.97

   [4] Acer - Nitro 15.6\" FHD IPS 144Hz Gaming Laptop -13th Gen In...
       💰 $879.99


In [11]:
# Cell 11: Step 4 - Select Top Deals with GPT-5-mini

print("="*60)
print(f"🤖 STEP 4: Selecting top 5 deals with GPT-5-mini")
print("="*60)

scanner = BestBuyScannerAgent()
deal_selection = scanner.scan(scraped_deals)

if deal_selection and deal_selection.deals:
    print(f"\n✅ Selected {len(deal_selection.deals)} best deals:")
    for i, deal in enumerate(deal_selection.deals, 1):
        print(f"\n   [{i}] {deal.product_description[:80]}...")
        print(f"       💰 ${deal.price}")
else:
    print("\n⚠️ No deals selected. Check scraped_deals.")

INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is initializing
INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is ready
INFO:root:[BestBuy Scanner Agent] Calling gpt-5-mini with 4 deals...


🤖 STEP 4: Selecting top 5 deals with GPT-5-mini


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[BestBuy Scanner Agent] Selected 4 deals



✅ Selected 4 best deals:

   [1] The Acer Nitro 15.6" is a gaming laptop built around a 13th Gen Intel Core i7 pr...
       💰 $879.99

   [2] The Acer Nitro 17.3" is a performance-focused gaming laptop powered by a 12th Ge...
       💰 $279.99

   [3] The Acer Aspire Lite 15 features a 15.6" Full HD IPS display that delivers crisp...
       💰 $329.99

   [4] The Acer Aspire 3 A315-58-7138 is a 15.6" Full HD laptop powered by an Intel Cor...
       💰 $638.97


In [12]:
# Cell 12: Initialize EnsembleAgent for Price Estimation

print("="*60)
print("🧠 Initializing EnsembleAgent (3 models)")
print("="*60)

import chromadb
from price_agents.ensemble_agent import EnsembleAgent

# Connect to ChromaDB
DB_PATH = "products_vectorstore"
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection('products')

print(f"ChromaDB: {collection.count()} documents")

# Initialize EnsembleAgent
ensemble = EnsembleAgent(collection)
print("\n✅ EnsembleAgent ready!")

🧠 Initializing EnsembleAgent (3 models)


INFO:datasets:PyTorch version 2.9.0 available.
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI


ChromaDB: 800000 documents


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready



✅ EnsembleAgent ready!


In [13]:
# Cell 13: Step 5 - Estimate Prices & Calculate Discounts

print("="*60)
print(f"💰 STEP 5: Estimating prices with EnsembleAgent")
print("="*60)

opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f"\n[{i}/{len(deal_selection.deals)}] Estimating: {deal.product_description[:50]}...")
    
    # Get price estimate from EnsembleAgent
    estimate = ensemble.price(deal.product_description)
    discount = estimate - deal.price
    
    # Create Opportunity
    opportunity = Opportunity(
        deal=deal,
        estimate=estimate,
        discount=discount
    )
    opportunities.append(opportunity)
    
    print(f"   💵 Sale Price:  ${deal.price:.2f}")
    print(f"   📊 Estimate:    ${estimate:.2f}")
    print(f"   🏷️  Discount:    ${discount:.2f}")

# Sort by discount (highest first)
opportunities.sort(key=lambda x: x.discount, reverse=True)

print(f"\n✅ Estimated {len(opportunities)} opportunities!")

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
10:22:32 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


💰 STEP 5: Estimating prices with EnsembleAgent

[1/4] Estimating: The Acer Nitro 15.6" is a gaming laptop built arou...


10:22:33 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $999.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $1199.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $913.20
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $1150.42
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
10:23:11 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


...

   💵 Sale Price:  $329.99
   📊 Estimate:    $337.99
   🏷️  Discount:    $8.00

[4/4] Estimating: The Acer Aspire 3 A315-58-7138 is a 15.6" Full HD ...


10:23:15 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $680.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $899.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $696.03
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $856.80


   💵 Sale Price:  $638.97
   📊 Estimate:    $856.80
   🏷️  Discount:    $217.83

✅ Estimated 4 opportunities!


In [14]:
# Cell 14: Final Results - Display Table

print("="*80)
print("🏆 FINAL RESULTS - Best Deals (Sorted by Discount)")
print("="*80)
print(f"\n📝 Original keyword: '{TEST_KEYWORD}'")
print(f"🔍 Refined query:    '{SEARCH_QUERY}'")
print(f"📊 Found {len(opportunities)} deals\n")

for i, opp in enumerate(opportunities, 1):
    discount_pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    
    # Status emoji based on discount
    if opp.discount > 200:
        status = "🔥 HOT DEAL!"
    elif opp.discount > 100:
        status = "✅ Good Deal"
    elif opp.discount > 0:
        status = "👍 OK"
    else:
        status = "❌ Overpriced"
    
    print(f"{'─'*80}")
    print(f"#{i} {status}")
    print(f"   📦 {opp.deal.product_description[:70]}...")
    print(f"   💵 Sale Price:     ${opp.deal.price:.2f}")
    print(f"   📊 Estimated Value: ${opp.estimate:.2f}")
    print(f"   🏷️  Discount:        ${opp.discount:.2f} ({discount_pct:.1f}%)")
    print(f"   🔗 {opp.deal.url}")

print(f"\n{'='*80}")
print(f"📊 Summary:")
print(f"   - Total deals: {len(opportunities)}")
print(f"   - Positive discount: {len([o for o in opportunities if o.discount > 0])}")
print(f"   - Great deals (>$100): {len([o for o in opportunities if o.discount > 100])}")
if opportunities:
    print(f"   - Best discount: ${opportunities[0].discount:.2f}")

🏆 FINAL RESULTS - Best Deals (Sorted by Discount)

📝 Original keyword: 'laptop'
🔍 Refined query:    'Acer laptops for students under $800'
📊 Found 4 deals

────────────────────────────────────────────────────────────────────────────────
#1 🔥 HOT DEAL!
   📦 The Acer Nitro 17.3" is a performance-focused gaming laptop powered by...
   💵 Sale Price:     $279.99
   📊 Estimated Value: $889.16
   🏷️  Discount:        $609.17 (68.5%)
   🔗 https://www.bestbuy.com/product/acer-nitro-5-17-3-full-hd-ips-144hz-gaming-laptop-intel-core-i5-12500h-nvidia-geforce-rtx-3050-512gb-pcie-gen-4-ssd-black/JJ8HLQL7YK
────────────────────────────────────────────────────────────────────────────────
#2 🔥 HOT DEAL!
   📦 The Acer Nitro 15.6" is a gaming laptop built around a 13th Gen Intel ...
   💵 Sale Price:     $879.99
   📊 Estimated Value: $1150.42
   🏷️  Discount:        $270.43 (23.5%)
   🔗 https://www.bestbuy.com/product/acer-nitro-15-6-fhd-ips-144hz-gaming-laptop-13th-gen-intel-core-i7-nvidia-geforce-rtx-40

In [ ]:
# Cell 15: (Optional) Send Push Notification for Best Deal
# Uncomment to send notification

# from price_agents.messaging_agent import MessagingAgent

# if opportunities and opportunities[0].discount > 50:
#     best = opportunities[0]
#     print("🔔 Sending push notification for best deal...")
#     
#     messenger = MessagingAgent()
#     messenger.notify(
#         description=best.deal.product_description[:200],
#         deal_price=best.deal.price,
#         estimated_true_value=best.estimate,
#         url=best.deal.url
#     )
#     print("✅ Notification sent!")
# else:
#     print("ℹ️ No deal with discount > $50")

print("\n🎉 FULL PIPELINE TEST COMPLETE!")
print("\nNext steps:")
print("1. Test with different keywords (TV, headphones, etc.)")
print("2. Test 'Skip' flow (user doesn't want clarification)")
print("3. Migrate to Gradio UI")